# 20. Review and average the paired study

Use a CPU runtime. This notebook downloads verified reports, checks pairing, and shows
every valid seed result alongside setting means and standard deviations. Invalid pairs
remain visible and are excluded from averages. Pending and resumable work is listed.

The overall average gives equal weight to settings with all planned seeds complete.
A partial summary is explicitly labeled. Shared controls and retained tasks are correlated,
so task rows and epoch measurements are not independent repetitions. Three seeds give
limited precision. No confidence interval or information-erasure claim is produced.

## 1. Load the fixed code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading this fixed code revision.")
if IN_COLAB:
    REPO = Path("/content/cark-paired-study")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has edits. Use a fresh runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a CARK checkout.")
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    implementation = ["uncle", "pyproject.toml", "requirements.txt"]
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--", *implementation], text=True).strip()
    # A later documentation-only commit may contain the same fixed implementation.
    differences = subprocess.check_output(["git", "-C", str(REPO), "diff", CODE_VERSION, actual, "--", *implementation], text=True).strip()
    if dirty or differences:
        raise RuntimeError("Use a clean checkout of CODE_VERSION, or open in Colab. Local training edits are preserved.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies before running this notebook.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))
from uncle.storage import R2Store
from uncle.paired_study import StudyStore, make_plan, jobs, inspect_queue, run_queue, review
import torch
print("Fixed implementation revision:", CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)


## 2. Select the study

In [ ]:
STUDY_ID = "paired_generalization_v1"
BUCKET = "uncle-experiments"
CLEAN_COMPLETED_RESUME_SLOTS = False  # Review is read-only until the final export cell.
SAVE_SUMMARY_TO_R2 = False
if torch.cuda.is_available():
    raise RuntimeError("Switch to a CPU runtime for analysis.")


## 3. Connect and inspect progress

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value
store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)


In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
path = store.download(PREFIX + "/study.json", CACHE / "cache")
plan = json.loads(path.read_text())
if plan["code_version"] != CODE_VERSION:
    raise ValueError("This study requires a different fixed code revision. Use its recorded notebook revision.")
from uncle.learning_diagnostic import file_hash
if file_hash(REPO / "uncle/task_partition.json") != plan["partition_sha256"]:
    raise ValueError("Class partition differs from the saved study.")
storage = StudyStore(store, plan, CACHE, cleanup_completed=CLEAN_COMPLETED_RESUME_SLOTS)
rows = inspect_queue(plan, storage, progress=True)
from collections import Counter
print("Queue:", dict(Counter(row["status"] for row in rows)))
for row in rows:
    print(row["status"].ljust(10), row["id"], row["active"] or "")


## 4. Read reports and compute paired effects

In [ ]:
result = review(plan, storage, progress=True)
print("Summary:", result["status"], "Valid pairs:", result["valid_pairs"], "/", result["expected_pairs"])
print("Complete settings:", result["complete_settings"])
print("Equal-setting means:", json.dumps(result["equal_setting_means"], indent=2))
print("Negative new-task differences:", result["new_task_negative_pairs"])
print("Within the fixed forgetting tolerance:", result["forgetting_within_tolerance_pairs"])
print("Relapse above that tolerance:", result["relapse_pairs"])
for row in result["invalid_pairs"]:
    print("INVALID PAIR:", row["branch_id"], row["pair_checks"])
for row in result["unfinished_jobs"]:
    print("UNFINISHED:", row["status"], row["job"])
for record in result["session_records"]:
    if record["status"] == "interrupted":
        print("INTERRUPTED SESSION:", record["session_id"], record.get("current_job"), record.get("error_type"))


## 5. Show seed results and individual retained-task changes

In [ ]:
for setting in result["settings"]:
    print("\n", setting["order"], "forget", setting["forget_task"], "then learn", setting["new_task"])
    print("Seeds:", setting["seeds"], "Complete setting:", setting["complete_setting"])
    for metric in ("new_task_difference", "retained_mean_difference", "largest_paired_retained_drop"):
        measurement = setting[metric]
        print(metric, "mean:", round(measurement["mean"], 3),
            "SD:", measurement["sd"], "values:", measurement["values"])
for row in result["comparisons"]:
    print(row["order"], "seed", row["seed"], "forget", row["forget_task"],
        "new-task difference:", row["new_task_difference"],
        "retained differences:", row["retained_differences"])
print(result["limitation"])


## 6. Plot the paired effects

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime, timezone
EXPORT = CACHE / "summaries" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
EXPORT.mkdir(parents=True, exist_ok=True)
if result["settings"]:
    labels = [s["order"] + "/U" + s["forget_task"] + "/L" + s["new_task"] for s in result["settings"]]
    fig, axes = plt.subplots(2, 1, figsize=(max(9, len(labels)*0.9), 8), sharex=True)
    for axis, metric, title in zip(axes,
        ("new_task_difference", "retained_mean_difference"),
        ("New-task accuracy: B minus A", "Mean retained accuracy: B minus A")):
        for index, setting in enumerate(result["settings"]):
            measurement = setting[metric]
            values = measurement["values"]
            offsets = [0.08*(i-(len(values)-1)/2) for i in range(len(values))]
            axis.scatter([index+o for o in offsets], values, color="#245d99", label="Seed result" if index==0 else None)
            axis.errorbar(index, measurement["mean"], yerr=measurement["sd"] or 0,
                fmt="s", color="#b85b22", capsize=4, label="Mean and SD" if index==0 else None)
        axis.axhline(0, color="black", linewidth=0.8)
        axis.set_ylabel("Percentage points")
        axis.set_title(title)
        axis.legend()
    axes[-1].set_xticks(range(len(labels)), labels, rotation=45, ha="right")
    fig.suptitle("Paired study: " + result["status"] + " (SD describes seed variation)")
    fig.tight_layout()
    fig.savefig(EXPORT / "paired_effects.png", dpi=180)
    fig.savefig(EXPORT / "paired_effects.svg")
    plt.show()
else:
    print("No valid completed pairs to plot yet.")


## 7. Save review artifacts

In [ ]:
import csv
from uncle.learning_diagnostic import _write_json
_write_json(EXPORT / "summary.json", result)
_write_json(EXPORT / "study.json", plan)
fields = ["status", "order", "seed", "forget_task", "new_task", "control_id", "branch_id",
    "new_task_a", "new_task_b", "new_task_difference", "retained_mean_difference",
    "starting_target_accuracy", "after_unlearning_accuracy", "final_target_accuracy",
    "target_start_above_chance", "forgetting_within_tolerance", "relapse_above_tolerance",
    "largest_paired_retained_drop", "largest_temporary_retained_drop"]
with (EXPORT / "comparisons.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(result["comparisons"])
with (EXPORT / "retained_tasks.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["status", "order", "seed", "forget_task", "new_task", "retained_task", "difference_pp"])
    writer.writeheader()
    for row in result["comparisons"]:
        for task, difference in row["retained_differences"].items():
            writer.writerow({**{key:row[key] for key in ("status","order","seed","forget_task","new_task")},
                "retained_task":task, "difference_pp":difference})
if SAVE_SUMMARY_TO_R2:
    from tqdm.auto import tqdm
    for path in tqdm(sorted(EXPORT.iterdir()), desc="Save review artifacts", unit="file"):
        store.upload(path, PREFIX + "/summaries/" + EXPORT.name + "/" + path.name)
    print("Review artifacts uploaded and verified.")
print("Local review artifacts:", EXPORT)
print("No models trained, modified, or deleted by this review.")
